In [ ]:
# ================= INSTALL =================
!apt-get install -y tesseract-ocr
!pip install pytesseract pillow

# ================= IMPORTS =================
import os
from PIL import Image
import cv2
import pytesseract
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
import numpy as np
from google.colab import files
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
import re
import random
from datetime import datetime
import gc

# ================= SETTINGS =================
IMG_SIZE = 224
BATCH_SIZE = 4

# ================= CLEAN DATASET =================
valid_ext = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')

def clean_dataset(directory):
    print(f"🔍 Cleaning {directory}...")

    for root, dirs, files in os.walk(directory):
        for file in files:

            path = os.path.join(root, file)

            # Remove unsupported files
            if not file.lower().endswith(valid_ext):
                print("❌ Removing unsupported:", path)
                os.remove(path)
                continue

            # Remove corrupted images
            try:
                img = Image.open(path)
                img.verify()

            except:
                print("❌ Removing corrupted:", path)
                os.remove(path)

# ================= DATASET PATHS =================
type_dataset = "/content/drive/MyDrive/dataset_type"
brand_dataset = "/content/drive/MyDrive/dataset_brand"

# ================= CLEAN DATASETS =================
clean_dataset(type_dataset)
clean_dataset(brand_dataset)

# ================= SAFE IMAGE LOADING =================
def load_images_safe(directory):

    images = []
    labels = []

    class_names = sorted(os.listdir(directory))

    for label, class_name in enumerate(class_names):

        class_path = os.path.join(directory, class_name)

        if not os.path.isdir(class_path):
            continue

        print(f"📂 Loading Class: {class_name}")

        for file in os.listdir(class_path):

            img_path = os.path.join(class_path, file)

            try:
                # Open image safely
                img = Image.open(img_path).convert("RGB")

                # Resize image
                img = img.resize((IMG_SIZE, IMG_SIZE))

                # Convert to array
                img_array = np.array(img, dtype=np.float32) / 255.0

                images.append(img_array)
                labels.append(label)

            except Exception as e:
                print("❌ Skipping:", img_path)
                print(e)

    images = np.array(images, dtype=np.float32)
    labels = np.array(labels)

    return images, labels, class_names

# ================= LOAD DATASETS =================
print("\n📦 Loading Vehicle Type Dataset...")
X_type, y_type, type_classes = load_images_safe(type_dataset)

gc.collect()

print("\n📦 Loading Vehicle Brand Dataset...")
X_brand, y_brand, brand_classes = load_images_safe(brand_dataset)

gc.collect()

print("\n✅ Type Dataset Shape:", X_type.shape)
print("✅ Brand Dataset Shape:", X_brand.shape)

# ================= MODEL FUNCTION =================
def create_model(num_classes):

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )

    base_model.trainable = False

    model = models.Sequential([

        base_model,

        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation='relu'),

        layers.Dropout(0.3),

        layers.Dense(num_classes, activation='softmax')

    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# ================= TRAIN TYPE MODEL =================
print("\n🚀 Training Vehicle Type Model...")

type_model = create_model(len(type_classes))

type_model.fit(
    X_type,
    y_type,
    epochs=5,
    batch_size=BATCH_SIZE
)

type_model.save("vehicle_type_model.h5")

print("✅ Vehicle Type Model Saved")

gc.collect()

# ================= TRAIN BRAND MODEL =================
print("\n🚀 Training Vehicle Brand Model...")

brand_model = create_model(len(brand_classes))

brand_model.fit(
    X_brand,
    y_brand,
    epochs=5,
    batch_size=BATCH_SIZE
)

brand_model.save("brand_model.h5")

print("✅ Brand Model Saved")

gc.collect()

print("\n🎉 ALL TRAINING COMPLETED SUCCESSFULLY")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
🔍 Cleaning /content/drive/MyDrive/dataset_type...
🔍 Cleaning /content/drive/MyDrive/dataset_brand...

📦 Loading Vehicle Type Dataset...
📂 Loading Class: auto
📂 Loading Class: car
📂 Loading Class: truck

📦 Loading Vehicle Brand Dataset...
📂 Loading Class: Daiatsu_Core
📂 Loading Class: Daiatsu_Mira
📂 Loading Class: FAW_V2
📂 Loading Class: Honda_BRV
📂 Loading Class: Honda_Grace
📂 Loading Class: Honda_Vezell
📂 Loading Class: Honda_city
📂 Loading Class: Honda_civic
📂 Loading Class: KIA_Sportage
📂 Loading Class: Suzuki_Mehran
📂 Loading Class: Suzuki_alto
📂 Loading Class: Suzuki_cultus
📂 Loading Class: Suzuki_kyber
📂 Loading Class: Suzuki_liana
📂 Loading Class: Suzuki_margala
📂 Loading Class: Suzuki_swift
📂 Loading Class: Suzuki_wagonR_2015
📂 Loading Class: Toyota_Aqua
📂 Lo

✅ Vehicle Type Model Saved

🚀 Training Vehicle Brand Model...
Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 162s 167ms/step - accuracy: 0.3341 - loss: 2.4015
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 150s 160ms/step - accuracy: 0.5000 - loss: 1.7251
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 149s 159ms/step - accuracy: 0.5888 - loss: 1.3880
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 152s 162ms/step - accuracy: 0.6331 - loss: 1.1921
Epoch 5/5
613/938 ━━━━━━━━━━━━━━━━━━━━ 52s 162ms/step - accuracy: 0.6832 - loss: 1.0094

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
# ================= TRAIN TOLL MODEL =================
data = {
    "vehicle_type": ["Car", "Truck", "Auto"],
    "distance": [30, 30, 30],
    "toll_fee": [80, 100, 150]
}

df = pd.DataFrame(data)

encoder = LabelEncoder()
df["vehicle_type_enc"] = encoder.fit_transform(df["vehicle_type"])

X = df[["vehicle_type_enc", "distance"]]
y = df["toll_fee"]

toll_model = LinearRegression()
toll_model.fit(X, y)

print("✅ All models trained successfully")
print()
print()
# ================= LOAD MODELS =================
type_model = tf.keras.models.load_model("vehicle_type_model.h5",compile=False)
brand_model = tf.keras.models.load_model("brand_model.h5",compile=False)
# ================= UPLOAD IMAGE =================
uploaded = files.upload()
img_path = list(uploaded.keys())[0]
print("📂 Uploaded:", img_path)

# ================= OCR =================
# ================= OCR =================
img = cv2.imread(img_path)

if img is None:
    raise ValueError("❌ Invalid image uploaded!")

# Resize image
img = cv2.resize(img, (800, 600))

# Convert to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Noise removal
gray = cv2.bilateralFilter(gray, 11, 17, 17)

# Thresholding
thresh = cv2.threshold(
    gray,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)[1]

# OCR configuration
custom_config = r'--oem 3 --psm 8 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'

# Extract text
text = pytesseract.image_to_string(
    thresh,
    config=custom_config
)

# Clean number plate
vehicle_number = re.sub(r'[^A-Z0-9]', '', text.upper())

print()
print("----------------------TOLL RECIEPT----------------------")
print()

if len(vehicle_number) >= 4:
    print("          Vehicle Number : ", vehicle_number)
else:
    print("          Vehicle Number : Not detected properly")

print()



# ================= PREDICTION =================
def predict_vehicle(img_path):
    try:
        img = image.load_img(img_path, target_size=(224, 224))
    except:
        raise ValueError("❌ Error loading image")

    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    type_pred = type_model.predict(img_array,verbose=0)
    vehicle_type = type_classes[np.argmax(type_pred)]
     # ================= ENTRY =================
    entry_time = datetime.now()

    print("          Entry Time     : ", entry_time)
    print()
    print("          Vehicle Type   : ", vehicle_type)
    print()

    # ================= EXIT + TOLL =================
    distance = random.randint(20, 100)

    try:
        vehicle_type_enc = encoder.transform([vehicle_type.capitalize()])[0]
    except:
        print("⚠️ Unknown vehicle type, defaulting to Car")
        vehicle_type_enc = encoder.transform(["Car"])[0]

    predicted_fee = toll_model.predict([[vehicle_type_enc,distance]])[0]

    exit_time = datetime.now()

    if vehicle_type.lower() == "car":
        brand_pred = brand_model.predict(img_array,verbose=0)
        brand = brand_classes[np.argmax(brand_pred)]
        print("          Car Brand    : ", brand)
        print()
    else:
        print("          Vehicle Brand  : Heavy Vehicle")
        print()

    return vehicle_type, distance, predicted_fee, exit_time

vehicle_type, distance, predicted_fee, exit_time = predict_vehicle(img_path)


print("          Distance       : ", distance, "km")
print()
print("          Toll Fee       : ₹", round(predicted_fee, 2))
print()
print("          Exit Time      : ", exit_time)
print()
print("---------------------------END---------------------------")
print()
print()
print()
print()
print()



NameError: name 'pd' is not defined